In [ ]:
#!/usr/bin/env python
# coding: utf-8

# # 多模态脑MRI数据集探索
# 
# 这个notebook用于探索和验证多模态脑MRI数据集的结构完整性。

# ## 1. 导入必要的库

import os
from pathlib import Path
import pandas as pd
from datetime import datetime
import json

# ## 2. 设置数据路径和定义文件结构

# 根目录路径
ROOT_DIR = Path("/home/jovyan/gpu_space/workspace_jiayi/new_datasets/NEW_DATASET_ANALYSIS")

# 定义每个受试者应该包含的文件结构
REQUIRED_FILES = {
    "4D_image": "evaluated/realigned_coregistered/nibabel_stacked_normalized_skull_stripped.nii.gz",
    "3D_label": "seg/resampled_synthseg_t1_mp2rage.nii.gz"
}

# ## 3. 扫描和验证数据集

def scan_dataset(root_dir):
    """
    扫描根目录，找出所有符合条件的受试者文件夹并验证文件完整性
    
    Parameters:
    -----------
    root_dir : Path
        数据集根目录路径
    
    Returns:
    --------
    dict : 包含扫描结果的字典
    """
    
    results = {
        "scan_time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "root_directory": str(root_dir),
        "total_subjects": 0,
        "valid_subjects": 0,
        "incomplete_subjects": 0,
        "subjects": []
    }
    
    # 检查根目录是否存在
    if not root_dir.exists():
        print(f"❌ 错误：根目录不存在 - {root_dir}")
        return results
    
    print(f"📁 扫描目录: {root_dir}\n")
    print("=" * 80)
    
    # 查找所有FOR_开头的文件夹
    subject_folders = [f for f in root_dir.iterdir() 
                      if f.is_dir() and f.name.startswith("FOR_")]
    
    results["total_subjects"] = len(subject_folders)
    
    if not subject_folders:
        print("⚠️ 警告：未找到任何以'FOR_'开头的文件夹")
        return results
    
    print(f"🔍 找到 {len(subject_folders)} 个受试者文件夹\n")
    
    # 检查每个受试者文件夹
    for idx, subject_folder in enumerate(subject_folders, 1):
        subject_info = {
            "subject_id": subject_folder.name,
            "path": str(subject_folder),
            "status": "完整",
            "missing_files": [],
            "existing_files": {}
        }
        
        print(f"[{idx}/{len(subject_folders)}] 检查受试者: {subject_folder.name}")
        
        # 检查必需的文件
        all_files_exist = True
        for file_type, relative_path in REQUIRED_FILES.items():
            file_path = subject_folder / relative_path
            
            if file_path.exists():
                subject_info["existing_files"][file_type] = str(file_path)
                print(f"  ✓ {file_type}: 找到")
                
                # 获取文件大小
                file_size_mb = file_path.stat().st_size / (1024 * 1024)
                subject_info["existing_files"][f"{file_type}_size_mb"] = round(file_size_mb, 2)
                
            else:
                all_files_exist = False
                subject_info["missing_files"].append(file_type)
                subject_info["status"] = "不完整"
                print(f"  ✗ {file_type}: 缺失")
        
        if all_files_exist:
            results["valid_subjects"] += 1
            print(f"  状态: ✅ 完整\n")
        else:
            results["incomplete_subjects"] += 1
            print(f"  状态: ⚠️ 不完整 - 缺失 {len(subject_info['missing_files'])} 个文件\n")
        
        results["subjects"].append(subject_info)
    
    return results

# 执行扫描
print("🚀 开始扫描数据集...\n")
scan_results = scan_dataset(ROOT_DIR)

# ## 4. 显示汇总信息

print("\n" + "=" * 80)
print("📊 数据集汇总信息")
print("=" * 80)
print(f"扫描时间: {scan_results['scan_time']}")
print(f"根目录: {scan_results['root_directory']}")
print(f"\n📈 统计信息:")
print(f"  • 总受试者数: {scan_results['total_subjects']}")
print(f"  • 完整数据受试者数: {scan_results['valid_subjects']} ({scan_results['valid_subjects']/max(scan_results['total_subjects'], 1)*100:.1f}%)")
print(f"  • 不完整数据受试者数: {scan_results['incomplete_subjects']} ({scan_results['incomplete_subjects']/max(scan_results['total_subjects'], 1)*100:.1f}%)")

# ## 5. 创建详细的数据框

# 创建受试者信息数据框
subjects_data = []
for subject in scan_results["subjects"]:
    row = {
        "受试者ID": subject["subject_id"],
        "状态": subject["status"],
        "4D影像": "✓" if "4D_image" in subject["existing_files"] else "✗",
        "3D标签": "✓" if "3D_label" in subject["existing_files"] else "✗",
        "缺失文件数": len(subject["missing_files"])
    }
    
    # 添加文件大小信息（如果存在）
    if "4D_image_size_mb" in subject["existing_files"]:
        row["4D影像大小(MB)"] = subject["existing_files"]["4D_image_size_mb"]
    if "3D_label_size_mb" in subject["existing_files"]:
        row["3D标签大小(MB)"] = subject["existing_files"]["3D_label_size_mb"]
    
    subjects_data.append(row)

df_subjects = pd.DataFrame(subjects_data)

print("\n📋 受试者详细信息:")
print(df_subjects.to_string(index=False))

# ## 6. 提取有效受试者路径列表

valid_subject_paths = []
valid_subject_info = []

for subject in scan_results["subjects"]:
    if subject["status"] == "完整":
        valid_subject_paths.append(subject["path"])
        valid_subject_info.append({
            "subject_id": subject["subject_id"],
            "path": subject["path"],
            "4d_image_path": subject["existing_files"]["4D_image"],
            "3d_label_path": subject["existing_files"]["3D_label"]
        })

print(f"\n✅ 找到 {len(valid_subject_paths)} 个数据完整的受试者")
print("\n有效受试者路径列表:")
for i, path in enumerate(valid_subject_paths, 1):
    print(f"{i}. {path}")

# ## 7. 保存结果

# 保存扫描结果为JSON文件
output_dir = Path("./mri_dataset_analysis_results")
output_dir.mkdir(exist_ok=True)

# 保存完整扫描结果
scan_results_file = output_dir / f"scan_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
with open(scan_results_file, 'w', encoding='utf-8') as f:
    json.dump(scan_results, f, ensure_ascii=False, indent=2)

print(f"\n💾 完整扫描结果已保存到: {scan_results_file}")

# 保存有效受试者信息
valid_subjects_file = output_dir / "valid_subjects.json"
with open(valid_subjects_file, 'w', encoding='utf-8') as f:
    json.dump(valid_subject_info, f, ensure_ascii=False, indent=2)

print(f"💾 有效受试者信息已保存到: {valid_subjects_file}")

# 保存为CSV格式（便于在Excel中查看）
csv_file = output_dir / f"subjects_summary_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
df_subjects.to_csv(csv_file, index=False, encoding='utf-8-sig')
print(f"💾 受试者汇总表已保存到: {csv_file}")

# ## 8. 数据质量检查

print("\n" + "=" * 80)
print("🔍 数据质量检查")
print("=" * 80)

# 检查不完整的受试者
incomplete_subjects = [s for s in scan_results["subjects"] if s["status"] == "不完整"]

if incomplete_subjects:
    print(f"\n⚠️ 发现 {len(incomplete_subjects)} 个数据不完整的受试者:")
    for subject in incomplete_subjects:
        print(f"\n受试者: {subject['subject_id']}")
        print(f"缺失文件: {', '.join(subject['missing_files'])}")
else:
    print("\n✅ 所有受试者数据完整！")

# 文件大小统计（仅针对完整数据）
if valid_subject_info:
    print("\n📊 文件大小统计（仅完整数据）:")
    
    sizes_4d = [s["existing_files"].get("4D_image_size_mb", 0) 
                for s in scan_results["subjects"] 
                if s["status"] == "完整"]
    sizes_3d = [s["existing_files"].get("3D_label_size_mb", 0) 
                for s in scan_results["subjects"] 
                if s["status"] == "完整"]
    
    if sizes_4d:
        print(f"\n4D影像文件:")
        print(f"  • 平均大小: {sum(sizes_4d)/len(sizes_4d):.2f} MB")
        print(f"  • 最小/最大: {min(sizes_4d):.2f} / {max(sizes_4d):.2f} MB")
    
    if sizes_3d:
        print(f"\n3D标签文件:")
        print(f"  • 平均大小: {sum(sizes_3d)/len(sizes_3d):.2f} MB")
        print(f"  • 最小/最大: {min(sizes_3d):.2f} / {max(sizes_3d):.2f} MB")

# ## 9. 快速访问有效数据的辅助函数

def get_subject_files(subject_id, valid_subjects=valid_subject_info):
    """
    根据受试者ID获取其文件路径
    
    Parameters:
    -----------
    subject_id : str
        受试者ID
    valid_subjects : list
        有效受试者信息列表
    
    Returns:
    --------
    dict : 包含文件路径的字典，如果未找到则返回None
    """
    for subject in valid_subjects:
        if subject["subject_id"] == subject_id:
            return {
                "4d_image": Path(subject["4d_image_path"]),
                "3d_label": Path(subject["3d_label_path"])
            }
    return None

# 示例：如何使用这个函数
if valid_subject_info:
    example_subject = valid_subject_info[0]["subject_id"]
    files = get_subject_files(example_subject)
    print(f"\n📌 示例：获取受试者 '{example_subject}' 的文件路径:")
    if files:
        print(f"  • 4D影像: {files['4d_image']}")
        print(f"  • 3D标签: {files['3d_label']}")

print("\n✨ 数据集探索完成！")
print(f"📁 所有结果已保存到: {output_dir.absolute()}")

# 将有效受试者路径列表存储为变量，便于后续使用
print(f"\n💡 提示：变量 'valid_subject_paths' 包含了所有数据完整的受试者路径")
print(f"        变量 'valid_subject_info' 包含了详细的文件路径信息")

In [ ]:
#!/usr/bin/env python
# coding: utf-8

# # NIfTI数据检查与分析
# 
# 这个notebook用于读取和检查多模态脑MRI数据的NIfTI文件

# ## 1. 导入必要的库

import nibabel as nib
import numpy as np
import pandas as pd
from pathlib import Path
import json
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# 设置显示选项
np.set_printoptions(precision=4, suppress=True)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# ## 2. 加载之前保存的有效受试者信息

# 加载有效受试者信息
valid_subjects_file = Path("./mri_dataset_analysis_results/valid_subjects.json")

if not valid_subjects_file.exists():
    print("❌ 错误：找不到valid_subjects.json文件")
    print("请先运行数据集探索notebook")
else:
    with open(valid_subjects_file, 'r', encoding='utf-8') as f:
        valid_subject_info = json.load(f)
    
    print(f"✅ 成功加载 {len(valid_subject_info)} 个有效受试者信息")

# ## 3. 读取第一个受试者的数据

if valid_subject_info:
    # 获取第一个受试者
    first_subject = valid_subject_info[0]
    print(f"\n📊 分析受试者: {first_subject['subject_id']}")
    print("=" * 80)
    
    # 读取4D影像和3D标签
    print("\n🔄 加载NIfTI文件...")
    
    # 4D影像
    img_4d_path = Path(first_subject['4d_image_path'])
    img_4d = nib.load(img_4d_path)
    data_4d = img_4d.get_fdata()
    
    print(f"✓ 4D影像加载完成: {img_4d_path.name}")
    
    # 3D标签
    label_3d_path = Path(first_subject['3d_label_path'])
    label_3d = nib.load(label_3d_path)
    data_label = label_3d.get_fdata()
    
    print(f"✓ 3D标签加载完成: {label_3d_path.name}")

# ## 4. 检查数据形状

print("\n📐 数据形状检查:")
print("=" * 80)

# 4D影像形状
expected_4d_shape = (288, 336, 384, 42)
actual_4d_shape = data_4d.shape
shape_match_4d = actual_4d_shape == expected_4d_shape

print(f"\n4D影像:")
print(f"  • 期望形状: {expected_4d_shape}")
print(f"  • 实际形状: {actual_4d_shape}")
print(f"  • 匹配状态: {'✅ 匹配' if shape_match_4d else '❌ 不匹配'}")

# 3D标签形状
expected_3d_shape = (288, 336, 384)
actual_3d_shape = data_label.shape
shape_match_3d = actual_3d_shape == expected_3d_shape

print(f"\n3D标签:")
print(f"  • 期望形状: {expected_3d_shape}")
print(f"  • 实际形状: {actual_3d_shape}")
print(f"  • 匹配状态: {'✅ 匹配' if shape_match_3d else '❌ 不匹配'}")

# ## 5. 检查头信息

print("\n🔍 NIfTI头信息:")
print("=" * 80)

# 4D影像头信息
print("\n📊 4D影像头信息:")
print(f"  • 数据类型: {img_4d.header.get_data_dtype()}")
print(f"  • 体素尺寸: {img_4d.header.get_zooms()[:3]} mm")
print(f"  • TR (如果可用): {img_4d.header.get_zooms()[3] if len(img_4d.header.get_zooms()) > 3 else 'N/A'} s")
print(f"  • 数据方向: {nib.aff2axcodes(img_4d.affine)}")

print("\n  • 仿射矩阵:")
print(img_4d.affine)

# 3D标签头信息
print("\n📊 3D标签头信息:")
print(f"  • 数据类型: {label_3d.header.get_data_dtype()}")
print(f"  • 体素尺寸: {label_3d.header.get_zooms()} mm")
print(f"  • 数据方向: {nib.aff2axcodes(label_3d.affine)}")

print("\n  • 仿射矩阵:")
print(label_3d.affine)

# 检查仿射矩阵是否匹配
affine_match = np.allclose(img_4d.affine, label_3d.affine, rtol=1e-5)
print(f"\n⚡ 仿射矩阵匹配: {'✅ 是' if affine_match else '❌ 否'}")

# ## 6. 4D影像数据统计分析

print("\n📊 4D影像数据统计:")
print("=" * 80)

# 计算每个时间点的统计信息
stats_per_timepoint = []
for t in range(data_4d.shape[3]):
    volume = data_4d[:, :, :, t]
    stats = {
        '时间点': t,
        '最小值': np.min(volume),
        '最大值': np.max(volume),
        '均值': np.mean(volume),
        '标准差': np.std(volume),
        '中位数': np.median(volume)
    }
    stats_per_timepoint.append(stats)

# 创建统计数据框
df_stats = pd.DataFrame(stats_per_timepoint)

print("\n各时间点统计信息摘要:")
print(df_stats.describe())

# 检查是否已经进行了Z-score标准化
print("\n🔍 检查数据标准化状态:")
global_mean = np.mean(data_4d)
global_std = np.std(data_4d)
print(f"  • 全局均值: {global_mean:.6f}")
print(f"  • 全局标准差: {global_std:.6f}")

# 判断是否接近标准正态分布
is_z_scored = abs(global_mean) < 0.1 and abs(global_std - 1.0) < 0.1
print(f"  • Z-score标准化: {'✅ 可能已标准化' if is_z_scored else '❌ 未标准化'}")

# 检查每个通道的分布
print("\n📈 前5个时间点的详细统计:")
print(df_stats.head())

# ## 7. 3D标签数据分析

print("\n🏷️ 3D标签数据分析:")
print("=" * 80)

# 获取唯一标签值
unique_labels = np.unique(data_label)
print(f"\n发现 {len(unique_labels)} 个唯一标签值")

# 统计每个标签的体素数量
label_counts = []
for label in unique_labels:
    count = np.sum(data_label == label)
    percentage = (count / data_label.size) * 100
    label_counts.append({
        '标签值': int(label),
        '体素数量': count,
        '占比(%)': percentage
    })

# 创建标签统计数据框
df_labels = pd.DataFrame(label_counts)
df_labels = df_labels.sort_values('标签值')

print("\n标签分布统计:")
print(df_labels.to_string(index=False))

# 计算总体素数
total_voxels = data_label.size
print(f"\n总体素数: {total_voxels:,}")
print(f"验证总和: {df_labels['体素数量'].sum():,} ({'✅ 匹配' if df_labels['体素数量'].sum() == total_voxels else '❌ 不匹配'})")

# ## 8. 检查标签的合理性

print("\n🔍 标签合理性检查:")
print("=" * 80)

# 定义常见的FreeSurfer/SynthSeg标签范围
# 这些是典型的脑区标签值，你可以根据实际情况调整
common_brain_labels = {
    0: "背景/CSF",
    2: "左侧大脑白质",
    3: "左侧大脑皮质",
    4: "左侧侧脑室",
    7: "左侧小脑白质", 
    8: "左侧小脑皮质",
    10: "左侧丘脑",
    11: "左侧尾状核",
    12: "左侧壳核",
    13: "左侧苍白球",
    14: "第三脑室",
    15: "第四脑室",
    16: "脑干",
    17: "左侧海马",
    18: "左侧杏仁核",
    24: "脑脊液",
    26: "左侧伏隔核",
    28: "左侧腹侧间脑",
    41: "右侧大脑白质",
    42: "右侧大脑皮质",
    43: "右侧侧脑室",
    46: "右侧小脑白质",
    47: "右侧小脑皮质",
    49: "右侧丘脑",
    50: "右侧尾状核",
    51: "右侧壳核",
    52: "右侧苍白球",
    53: "右侧海马",
    54: "右侧杏仁核",
    58: "右侧伏隔核",
    60: "右侧腹侧间脑"
}

# 检查是否有意外的标签值
expected_labels = set(common_brain_labels.keys())
actual_labels = set(int(label) for label in unique_labels)

# 找出意外的标签
unexpected_labels = actual_labels - expected_labels
missing_labels = expected_labels - actual_labels

if unexpected_labels:
    print(f"\n⚠️ 发现意外的标签值: {sorted(unexpected_labels)}")
else:
    print("\n✅ 所有标签值都在预期范围内")

# 显示标签含义（如果已知）
print("\n📋 标签含义对照表:")
for label in sorted(actual_labels):
    if label in common_brain_labels:
        voxel_count = df_labels[df_labels['标签值'] == label]['体素数量'].values[0]
        percentage = df_labels[df_labels['标签值'] == label]['占比(%)'].values[0]
        print(f"  {label:3d}: {common_brain_labels[label]:20s} - {voxel_count:8,d} 体素 ({percentage:5.2f}%)")
    else:
        voxel_count = df_labels[df_labels['标签值'] == label]['体素数量'].values[0]
        percentage = df_labels[df_labels['标签值'] == label]['占比(%)'].values[0]
        print(f"  {label:3d}: {'未知区域':20s} - {voxel_count:8,d} 体素 ({percentage:5.2f}%)")

# ## 9. 可视化标签分布

# 创建标签分布的柱状图
plt.figure(figsize=(12, 6))
plt.bar(df_labels['标签值'].astype(str), df_labels['占比(%)'])
plt.xlabel('标签值')
plt.ylabel('占比 (%)')
plt.title(f'标签分布 - {first_subject["subject_id"]}')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# ## 10. 数据质量总结

print("\n" + "=" * 80)
print("📋 数据质量总结")
print("=" * 80)

quality_checks = [
    ("4D影像形状", shape_match_4d),
    ("3D标签形状", shape_match_3d),
    ("仿射矩阵匹配", affine_match),
    ("标签值合理性", len(unexpected_labels) == 0),
    ("数据完整性", df_labels['体素数量'].sum() == total_voxels)
]

all_passed = True
for check_name, passed in quality_checks:
    status = "✅ 通过" if passed else "❌ 失败"
    print(f"{check_name}: {status}")
    all_passed = all_passed and passed

print("\n" + "=" * 80)
if all_passed:
    print("✨ 总体评估: 数据质量良好，所有检查项都通过！")
else:
    print("⚠️ 总体评估: 发现一些问题，请检查上述失败项。")

# ## 11. 保存检查结果

# 准备检查结果字典
check_results = {
    "subject_id": first_subject["subject_id"],
    "check_time": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
    "4d_shape": list(actual_4d_shape),
    "3d_shape": list(actual_3d_shape),
    "shape_match": {
        "4d": shape_match_4d,
        "3d": shape_match_3d
    },
    "data_stats": {
        "global_mean": float(global_mean),
        "global_std": float(global_std),
        "is_z_scored": is_z_scored
    },
    "label_info": {
        "unique_labels": [int(l) for l in unique_labels],
        "label_counts": df_labels.to_dict('records'),
        "unexpected_labels": list(unexpected_labels)
    },
    "quality_passed": all_passed
}

# 保存结果
output_file = Path("./mri_dataset_analysis_results/data_check_results.json")
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(check_results, f, ensure_ascii=False, indent=2)

print(f"\n💾 检查结果已保存到: {output_file}")

# ## 12. 批量检查所有受试者（可选）

print("\n" + "=" * 80)
print("🔄 批量检查所有受试者")
print("=" * 80)

batch_check = input("\n是否要检查所有受试者的数据一致性？(y/n): ")

if batch_check.lower() == 'y':
    all_subjects_results = []
    
    for subject in valid_subject_info:
        print(f"\n检查 {subject['subject_id']}...", end='')
        
        try:
            # 加载数据
            img = nib.load(subject['4d_image_path'])
            label = nib.load(subject['3d_label_path'])
            
            # 获取形状
            img_shape = img.shape
            label_shape = label.shape
            
            # 检查形状
            shape_ok = (img_shape == expected_4d_shape and 
                       label_shape == expected_3d_shape)
            
            result = {
                'subject_id': subject['subject_id'],
                '4d_shape': img_shape,
                '3d_shape': label_shape,
                'shape_ok': shape_ok
            }
            
            all_subjects_results.append(result)
            print(" ✓" if shape_ok else " ✗")
            
        except Exception as e:
            print(f" ❌ 错误: {str(e)}")
            all_subjects_results.append({
                'subject_id': subject['subject_id'],
                'error': str(e)
            })
    
    # 汇总结果
    df_batch = pd.DataFrame(all_subjects_results)
    
    print("\n批量检查结果汇总:")
    print(df_batch)
    
    # 保存批量检查结果
    batch_file = Path("./mri_dataset_analysis_results/batch_check_results.csv")
    df_batch.to_csv(batch_file, index=False)
    print(f"\n💾 批量检查结果已保存到: {batch_file}")

print("\n✅ 数据检查完成！")